# Proyecto Integrador · Minería de Datos Masivos
## Tareas 1 y 2 — Descarga, limpieza y esquema en Spark (SEACE / OCDS Perú)

**Fuente:** Registro OCDS Perú, OECE (ex OSCE), vía Open Contracting Partnership · https://data.open-contracting.org/es/publication/135 · Licencia CC BY 4.0 (datos modificados)

**Qué hace este notebook**
1. Toma los `.zip` de cada año desde `ZIPS_DIR` (o, si falta alguno, intenta descargarlo de la web).
2. Lee las 12 tablas que usamos con Spark y les da un esquema explícito.
3. Limpia, deduplica, filtra el periodo de análisis y excluye el último mes incompleto.
4. Deja **todas las fechas en hora de Lima** (UTC−5), sin zona horaria.
5. Agrega columnas derivadas: **ubicación de la entidad, estados** (proceso, adjudicación y contrato), **montos en soles** y tipo de entidad.
6. Exporta 6 tablas en **Parquet** para B y C.
7. Genera un reporte de calidad (para documentar la limpieza en el informe).

**Cómo usarlo**
- Pon `2024.zip`, `2025.zip` y `2026.zip` en la carpeta de Drive que indica `ZIPS_DIR`.
- Ejecuta todo (Entorno de ejecución → Ejecutar todo). Reescribe la carpeta `parquet/`.
- Para una prueba rápida, usa `ANIOS = [2026]`.
- Al final del notebook están los pasos para B y C (sección 10) y el diccionario de datos (sección 11).

## 0. Configuración

In [ ]:
# ── Parámetros principales ────────────────────────────────────
ANIOS = [2024, 2025, 2026]     # prueba rápida: [2026]
FECHA_PUB_MIN = "2024-01-01"   # periodo de análisis (por fecha de publicación). None = sin filtro
EXCLUIR_MES_PARCIAL = True     # quita el último mes del corte, que está incompleto (septiembre 2026: solo 3 días)

USAR_DRIVE = True              # True = monta Drive (zips de entrada y/o salida)
DRIVE_DIR  = "/content/drive/MyDrive/proyecto_integrador_seace"   # carpeta de salida (Parquet + reportes)

# Carpeta donde están 2024.zip, 2025.zip, 2026.zip (nombres exactos).
# Puede ser una carpeta de Drive, por ejemplo: "/content/drive/MyDrive/proyecto_integrador_seace/zips"
ZIPS_DIR   = "/content/drive/MyDrive/proyecto_integrador_seace/zips"

RAW_DIR    = "/content/raw"                  # CSV originales (solo las tablas que usamos)
SALIDA_DIR = "/content/salida"               # Parquet limpio + reportes

URL_ANIO = "https://data.open-contracting.org/es/publication/135/download?name={anio}.csv.tar.gz"

# Tablas que usamos (el resto del paquete OCDS se ignora: documentos, etc.)
TABLAS = [
    "main", "tender_items", "tender_items_additionalClassifications", "tender_tenderers",
    "awards", "awards_suppliers", "awards_items", "awards_value_exchangeRates",
    "contracts", "contracts_items", "contracts_value_exchangeRates",
    "parties",
]

In [ ]:
import os, json, shutil, subprocess, tarfile, zipfile, time
import requests
import pandas as pd

if USAR_DRIVE or ZIPS_DIR.startswith("/content/drive"):
    from google.colab import drive
    drive.mount("/content/drive")
if USAR_DRIVE:
    SALIDA_DIR = DRIVE_DIR

os.makedirs(RAW_DIR, exist_ok=True)
os.makedirs(SALIDA_DIR, exist_ok=True)
print("Años:", ANIOS, "| Pub. desde:", FECHA_PUB_MIN, "| Zips en:", ZIPS_DIR, "| Salida:", SALIDA_DIR)

Mounted at /content/drive
Años: [2024, 2025, 2026] | Pub. desde: 2024-01-01 | Zips en: /content/drive/MyDrive/proyecto_integrador_seace/zips | Salida: /content/drive/MyDrive/proyecto_integrador_seace


## 1. Spark

In [ ]:
# Java (Colab normalmente ya lo trae)
if shutil.which("java") is None:
    subprocess.run("apt-get -qq install -y openjdk-17-jre-headless", shell=True, check=True)

subprocess.run("pip -q install pyspark==3.5.3", shell=True, check=True)   # versión fija: compatible con Java 11/17
import pyspark

from pyspark.sql import SparkSession, functions as F, Window

spark = (SparkSession.builder
         .master("local[*]")
         .appName("seace_ocds")
         .config("spark.driver.memory", "8g")
         .config("spark.sql.shuffle.partitions", "16")        # 200 (default) es excesivo en local
         .config("spark.sql.session.timeZone", "America/Lima")  # las fechas se convierten a hora de Lima al leer
         .config("spark.sql.ansi.enabled", "false")           # casts tolerantes: valor inválido -> null
         .getOrCreate())
print("Spark", spark.version, "| núcleos:", spark.sparkContext.defaultParallelism)

Spark 3.5.3 | núcleos: 2


## 2. Tarea 1 — Descarga masiva

Cada año viene como un `.csv.tar.gz` (o un `.zip` equivalente) con **una tabla por archivo**, enlazadas por `main_ocid` / `main_id`.
Solo extraemos las 12 tablas que usamos (las demás son documentos, enlaces y otros datos que no se necesitan).

> **Ojo:** el año del archivo es el año de la **última actualización** del proceso, no el de su publicación.
> Por eso el archivo 2026 trae procesos antiguos (2013–2025) que se actualizaron en 2026.
> Más abajo filtramos por fecha de publicación.

In [ ]:
def _extraer_tablas_tar(ruta, destino):
    with tarfile.open(ruta, "r:gz") as tar:
        for mi in tar:
            base = os.path.basename(mi.name)
            if mi.isfile() and base.endswith(".csv") and base[:-4] in TABLAS and not base.startswith("._"):
                with tar.extractfile(mi) as src, open(f"{destino}/{base}", "wb") as out:
                    shutil.copyfileobj(src, out)

def _extraer_tablas_zip(ruta, destino):
    with zipfile.ZipFile(ruta) as z:
        for nombre in z.namelist():
            base = os.path.basename(nombre)
            if "__MACOSX" in nombre or base.startswith("._"):
                continue
            if base.endswith(".csv") and base[:-4] in TABLAS:
                with z.open(nombre) as src, open(f"{destino}/{base}", "wb") as out:
                    shutil.copyfileobj(src, out)

def preparar_anio(anio):
    destino = f"{RAW_DIR}/{anio}"
    if all(os.path.exists(f"{destino}/{t}.csv") for t in TABLAS):
        print(f"[{anio}] ya está listo.")
        return
    os.makedirs(destino, exist_ok=True)

    zip_local = f"{ZIPS_DIR}/{anio}.zip"
    if os.path.exists(zip_local):                      # 1) zip subido a mano
        print(f"[{anio}] usando {zip_local}")
        _extraer_tablas_zip(zip_local, destino)
    else:                                              # 2) descarga directa
        url, tmp = URL_ANIO.format(anio=anio), f"/content/_tmp_{anio}.csv.tar.gz"
        print(f"[{anio}] descargando {url}")
        t0 = time.time()
        r = requests.get(url, stream=True, timeout=60, headers={"User-Agent": "Mozilla/5.0"})
        if r.status_code != 200:
            raise RuntimeError(f"Descarga falló ({r.status_code}). Pon {anio}.zip en {ZIPS_DIR} y reintenta.")
        with open(tmp, "wb") as f:
            for chunk in r.iter_content(1 << 20):
                f.write(chunk)
        print(f"[{anio}] {os.path.getsize(tmp)/1e6:.0f} MB en {time.time()-t0:.0f}s, extrayendo...")
        _extraer_tablas_tar(tmp, destino)
        os.remove(tmp)

    faltan = [t for t in TABLAS if not os.path.exists(f"{destino}/{t}.csv")]
    if faltan:
        raise RuntimeError(f"[{anio}] faltan tablas: {faltan}")
    print(f"[{anio}] listo.")

for a in ANIOS:
    preparar_anio(a)

[2024] usando /content/drive/MyDrive/proyecto_integrador_seace/zips/2024.zip
[2024] listo.
[2025] usando /content/drive/MyDrive/proyecto_integrador_seace/zips/2025.zip
[2025] listo.
[2026] usando /content/drive/MyDrive/proyecto_integrador_seace/zips/2026.zip
[2026] listo.


In [ ]:
# Verificación rápida: tamaño de cada tabla por año
filas = []
for a in ANIOS:
    for t in TABLAS:
        filas.append({"anio": a, "tabla": t, "MB": round(os.path.getsize(f"{RAW_DIR}/{a}/{t}.csv")/1e6, 1)})
pd.DataFrame(filas).pivot(index="tabla", columns="anio", values="MB")

anio,2024,2025,2026
tabla,,,
awards,14.9,17.1,16.3
awards_items,40.0,46.9,44.4
awards_suppliers,15.1,17.4,16.7
awards_value_exchangeRates,0.3,0.4,0.3
contracts,35.3,43.7,37.8
contracts_items,33.4,39.9,36.5
contracts_value_exchangeRates,0.2,0.3,0.2
main,94.6,109.6,104.9
parties,325.4,438.2,385.4


## 3. Tarea 2 — Lectura con esquema explícito

**Decisión:** leemos todo como texto y luego **seleccionamos y convertimos por nombre**.
- No usamos `inferSchema` (es lento y puede variar entre años).
- Si un año trae columnas en otro orden, no se rompe.
- `multiLine=True` porque algunas descripciones tienen saltos de línea.
- Un valor que no se pueda convertir queda en `null` (no tumba el proceso).
- **Fechas:** el CSV trae `-05:00` en cada fecha. Se convierten a **hora local de Lima, sin zona horaria**
  (`timestamp_ntz`). Así se leen igual en Spark y pandas, sin conversiones.

In [ ]:
from functools import reduce

def leer(tabla, cols):
    """Lee `tabla` año por año y une por NOMBRE de columna.
    Cada archivo se lee con su propio encabezado (el orden de columnas puede variar entre años).
    `cols` = {nombre_csv: tipo_spark}."""
    partes = []
    for a in ANIOS:
        df = (spark.read.option("header", True).option("multiLine", True)
              .option("quote", '"').option("escape", '"').csv(f"{RAW_DIR}/{a}/{tabla}.csv"))
        faltan = [c for c in cols if c not in df.columns]
        if faltan:
            print(f"[aviso] {tabla} {a}: faltan columnas {faltan}")
        sel = []
        for c, t in cols.items():
            if c not in df.columns:
                sel.append(F.lit(None).cast("string" if t == "int" else t).alias(c)); continue
            e = F.col(c)
            if t == "int":
                e = e.cast("double").cast("int")                  # "11.0" -> 11
            elif t == "timestamp":
                e = e.cast("timestamp").cast("timestamp_ntz")     # instante -> hora local de Lima (sin zona)
            else:
                e = e.cast(t)
            sel.append(e.alias(c))
        partes.append(df.select(*sel).withColumn("anio_archivo", F.lit(a)))
    return reduce(lambda x, y: x.unionByName(y), partes)

def sin_cero(c):  # "8110151000336399.0" -> "8110151000336399"
    return F.regexp_replace(c.cast("string"), r"\.0+$", "")

def normalizar(c):
    """minúsculas, sin tildes, solo letras/números, espacios simples."""
    x = F.translate(F.lower(c), "áéíóúüñàèìòù", "aeiouunaeiou")
    x = F.regexp_replace(x, r"[^a-z0-9 ]", " ")
    return F.trim(F.regexp_replace(x, r"\s+", " "))

def ruc_de(c):
    return F.regexp_replace(c, r"^PE-RUC-", "")

def resumen_estado(df, claves, col):
    """Un estado por grupo: el valor único, "MIXTO" si hay más de uno, "sin_dato" si no hay."""
    g = df.groupBy(*claves).agg(F.countDistinct(col).alias("_n"), F.max(col).alias("_e"))
    return (g.withColumn("estado", F.when(F.col("_n") == 1, F.col("_e"))
                                    .when(F.col("_n") > 1, F.lit("MIXTO"))
                                    .otherwise(F.lit("sin_dato")))
             .drop("_n", "_e"))

def tipo_entidad(nombre):
    """Tipo de entidad según palabras del nombre. Es HEURÍSTICO (no viene en la fuente)."""
    n = normalizar(nombre)
    return (F.when(n.rlike("municipalidad"), "municipalidad")
             .when(n.rlike("gobierno regional"), "gobierno_regional")
             .when(n.rlike("ministerio"), "ministerio")
             .when(n.rlike("universidad"), "universidad")
             .when(n.rlike("hospital|salud|essalud|seguro social|red asistencial"), "salud")
             .when(n.rlike("ejercito|fuerza aerea|marina de guerra|policia|comando"), "defensa_policia")
             .otherwise("otro"))

In [ ]:
# Columnas que conservamos de cada tabla (y su tipo)
COLS_MAIN = {
    "id": "string", "ocid": "string", "date": "timestamp",
    "buyer_id": "string", "buyer_name": "string", "tender_id": "string",
    "tender_title": "string", "tender_description": "string",
    "tender_procurementMethod": "string", "tender_procurementMethodDetails": "string",
    "tender_mainProcurementCategory": "string",
    "tender_datePublished": "timestamp",
    "tender_value_amount": "double", "tender_value_currency": "string", "tender_value_amount_PEN": "double",
    "tender_tenderPeriod_startDate": "timestamp", "tender_tenderPeriod_endDate": "timestamp",
    "tender_numberOfTenderers": "int", "planning_budget_projectID": "string",
    "tender_hasTenderInformationProtectedByLaw": "boolean",
}
# Se descartan: tag/initiationType (constantes), tender_tenderPeriod_durationInDays (100% nulo),
# tender_procuringEntity_* (idéntico a buyer_*), enquiryPeriod_*, dataSegmentation_*, planning_budget_description.

COLS_ITEMS = {
    "id": "string", "description": "string", "quantity": "double", "unit_name": "string",
    "totalValue_amount": "double", "totalValue_currency": "string",
    "classification_id": "string", "classification_description": "string",
    "status": "string", "statusDetails": "string",
    "main_id": "string", "main_ocid": "string",
}
COLS_UNSPSC = {"id": "string", "description": "string", "main_id": "string", "tender_items_id": "string"}
COLS_TENDERERS = {"id": "string", "name": "string", "main_id": "string", "main_ocid": "string"}
COLS_AWARDS = {"id": "string", "date": "timestamp", "value_amount": "double", "value_currency": "string",
               "main_id": "string", "main_ocid": "string"}
COLS_SUPPLIERS = {"id": "string", "name": "string", "main_id": "string", "awards_id": "string"}
COLS_CONTRACTS = {
    "id": "string", "dateSigned": "timestamp", "awardID": "string",
    "value_amount": "double", "value_currency": "string",
    "period_startDate": "timestamp", "period_endDate": "timestamp", "period_durationInDays": "int",
    "implementation_finalValue_amount": "double", "implementation_finalValue_currency": "string",
    "status": "string", "statusDetails": "string",
    "main_id": "string", "main_ocid": "string",
}

# Tablas nuevas: estados, tipo de cambio y ubicación de la entidad
COLS_AWARD_ITEMS = {"awards_id": "string", "statusDetails": "string", "main_id": "string"}
COLS_CONTRACT_ITEMS = {"contracts_id": "string", "statusDetails": "string", "main_id": "string"}
COLS_FX_AWARDS = {"awards_id": "string", "rate": "double", "main_id": "string"}
COLS_FX_CONTRACTS = {"contracts_id": "string", "rate": "double", "main_id": "string"}
COLS_PARTIES = {"roles": "string", "address_department": "string", "address_region": "string",
                "address_locality": "string", "main_id": "string"}
# Se descartan de parties: teléfonos, correos y direcciones de calle (no se necesitan). Solo se usan las filas "buyer".

## 4. Limpieza de `main` → procesos

Pasos (cada uno queda contado en el reporte de calidad):
1. **Deduplicar por `ocid`** (si un proceso está en 2 años, queda la versión más reciente).
2. **Filtrar el periodo** por fecha de publicación (`FECHA_PUB_MIN`).
3. Fechas en hora de Lima, categorías sin nulos, texto normalizado.
4. Valores `0` en el monto: se tratan como valor no disponible, no como monto real. El 78% son procesos con
   **información protegida por ley** (`info_protegida`). Se marcan (`valor_cero`) y no entran en `valor_pen_valido`.
5. **Excluir el mes parcial:** el último mes del corte está incompleto (pocos días), así que se quita.
   Los datos van del primer mes del periodo hasta el último mes completo.
6. **Tipo de entidad** (`entidad_tipo`): se deduce de palabras del nombre (municipalidad, gobierno regional, ministerio...).
   Es un dato **heurístico**: no viene en la fuente y el 23% queda como `otro`.

In [ ]:
CALIDAD = {}   # aquí se acumulan los conteos para el reporte

m_raw = leer("main", COLS_MAIN).cache()
CALIDAD["main_filas_leidas"] = m_raw.count()
CALIDAD["main_filas_por_archivo"] = {int(r["anio_archivo"]): r["n"] for r in m_raw.groupBy("anio_archivo").agg(F.count("*").alias("n")).collect()}

# ── Chequeo de alineación: detecta columnas corridas entre años ──
chk = (m_raw.groupBy("anio_archivo").agg(
          F.count("*").alias("filas"),
          F.round(100 * F.avg(F.col("tender_datePublished").isNull().cast("int")), 2).alias("pct_fecha_pub_nula"),
          F.round(100 * F.avg(F.col("tender_value_amount_PEN").isNull().cast("int")), 2).alias("pct_valor_pen_nulo"),
          F.countDistinct("tender_mainProcurementCategory").alias("n_categorias"))
       .orderBy("anio_archivo").toPandas())
print(chk.to_string(index=False))
assert (chk.pct_fecha_pub_nula < 5).all() and (chk.n_categorias <= 4).all(),     "Columnas desalineadas en algún año: revisa el encabezado de main.csv de ese año"
print("Chequeo de alineación: OK")

# 1) Deduplicar por ocid (queda la versión más reciente)
w = Window.partitionBy("ocid").orderBy(F.col("date").desc())
m = m_raw.withColumn("_rn", F.row_number().over(w)).filter("_rn = 1").drop("_rn")
CALIDAD["main_ocid_duplicados_eliminados"] = CALIDAD["main_filas_leidas"] - m.count()

# 2) Periodo de análisis
if FECHA_PUB_MIN:
    n_antes = m.count()
    m = m.filter(F.col("tender_datePublished") >= F.lit(FECHA_PUB_MIN).cast("timestamp_ntz"))
    CALIDAD["main_fuera_de_periodo"] = n_antes - m.count()
else:
    CALIDAD["main_fuera_de_periodo"] = 0

# 2b) Mes parcial: el último mes del corte está incompleto -> se excluye
FECHA_CORTE = m.agg(F.max("date")).first()[0]     # última actualización vista en los datos
periodo_corte = FECHA_CORTE.strftime("%Y-%m")
if EXCLUIR_MES_PARCIAL:
    n_antes = m.count()
    m = m.filter(F.col("tender_datePublished") < F.lit(periodo_corte + "-01").cast("timestamp_ntz"))
    CALIDAD["main_mes_parcial_excluido"] = n_antes - m.count()
    CALIDAD["mes_parcial_excluido"] = periodo_corte
else:
    CALIDAD["main_mes_parcial_excluido"] = 0
    CALIDAD["mes_parcial_excluido"] = None

# 3) Renombrar + derivadas

m = (m.select(
        F.col("ocid"), F.col("id").alias("release_id"), F.col("date").alias("fecha_release"),
        F.col("tender_id").alias("proceso_id"),
        F.col("buyer_id").alias("entidad_id"), F.col("buyer_name").alias("entidad"),
        F.col("tender_title").alias("titulo"), F.col("tender_description").alias("descripcion"),
        F.coalesce(F.trim("tender_procurementMethod"), F.lit("sin_dato")).alias("metodo"),
        F.coalesce(F.trim("tender_procurementMethodDetails"), F.lit("sin_dato")).alias("procedimiento"),
        F.coalesce(F.trim("tender_mainProcurementCategory"), F.lit("sin_dato")).alias("categoria"),
        F.col("tender_datePublished").alias("fecha_pub"),
        F.col("tender_tenderPeriod_startDate").alias("fecha_inicio_conv"),
        F.col("tender_tenderPeriod_endDate").alias("fecha_fin_conv"),
        F.col("tender_value_amount").alias("valor_orig"), F.col("tender_value_currency").alias("moneda"),
        F.col("tender_value_amount_PEN").alias("valor_pen"),
        F.col("tender_numberOfTenderers").alias("n_postores_declarado"),
        F.col("planning_budget_projectID").alias("proyecto_id"),
        F.coalesce(F.col("tender_hasTenderInformationProtectedByLaw"), F.lit(False)).alias("info_protegida"),
        F.col("anio_archivo").alias("archivo_origen"),
     )
     .withColumn("anio_pub", F.year("fecha_pub"))
     .withColumn("mes_pub", F.month("fecha_pub"))
     .withColumn("periodo", F.date_format("fecha_pub", "yyyy-MM"))
     .withColumn("valor_cero", F.col("valor_pen") == 0)
     .withColumn("valor_pen_valido", F.when(F.col("valor_pen") > 0, F.col("valor_pen")))
     .withColumn("texto", normalizar(F.col("descripcion")))
     .withColumn("n_palabras", F.when(F.col("texto") == "", 0).otherwise(F.size(F.split("texto", " "))))
     .withColumn("entidad_tipo", tipo_entidad(F.col("entidad")))
)

# Guardar base y volver a leerla desde Parquet (evita re-leer los CSV pesados)
PQ = f"{SALIDA_DIR}/parquet"
shutil.rmtree(PQ, ignore_errors=True)
m.write.mode("overwrite").parquet(f"{PQ}/_procesos_base")
m_raw.unpersist()
base = spark.read.parquet(f"{PQ}/_procesos_base").cache()
CALIDAD["procesos_final"] = base.count()
CALIDAD["procesos_valor_cero"] = base.filter("valor_cero").count()
CALIDAD["procesos_info_protegida"] = base.filter("info_protegida").count()
CALIDAD["procesos_valor_cero_y_protegido"] = base.filter("valor_cero AND info_protegida").count()
CALIDAD["fecha_corte"] = str(FECHA_CORTE)
rango = base.agg(F.min("periodo"), F.max("periodo")).first()
CALIDAD["periodo_analisis"] = f"{rango[0]} a {rango[1]}"
print("Procesos finales:", CALIDAD["procesos_final"], "| corte:", FECHA_CORTE,
      "| mes parcial excluido:", CALIDAD["mes_parcial_excluido"], "| periodo:", CALIDAD["periodo_analisis"])

 anio_archivo  filas  pct_fecha_pub_nula  pct_valor_pen_nulo  n_categorias
         2024 109722                 0.0                 0.0             3
         2025 125451                 0.0                 0.0             3
         2026 122556                 0.0                 0.0             3
Chequeo de alineación: OK
Procesos finales: 209740 | corte: 2026-09-03 10:51:02.475118 | mes parcial excluido: 2026-09 | periodo: 2024-01 a 2026-08


## 5. Tablas hijas (ítems, postores, adjudicaciones, contratos, entidades)

Todas se filtran con un `left_semi` join contra los `release_id` que quedaron en `procesos`.
Así no queda ninguna fila huérfana.

Además de limpiar, aquí se agregan:
- **Estados** de ítems, adjudicaciones y contratos. El estado oficial del contrato viene vacío en la fuente, pero se deduce de los ítems del contrato.
- **Montos en soles** (`monto_pen`) para adjudicaciones y contratos, con el tipo de cambio que publica la fuente.
- **Ubicación de la entidad contratante** (departamento, provincia y distrito), tomada de `parties` solo en las filas "buyer".
  Se descartan teléfonos, correos y direcciones de calle.

In [ ]:
ids = F.broadcast(base.select(F.col("release_id").alias("_rid")))
def vigentes(df):
    return df.join(ids, df["main_id"] == F.col("_rid"), "left_semi")

# ── Ítems (con clasificación CUBSO/UNSPSC y estado) ───────────
it = vigentes(leer("tender_items", COLS_ITEMS))
un = (vigentes(leer("tender_items_additionalClassifications", COLS_UNSPSC))
      .groupBy("main_id", F.col("tender_items_id").alias("item_id_u"))
      .agg(F.first("id").alias("unspsc_id"), F.first("description").alias("unspsc_desc")))

items = (it.join(un, (it["main_id"] == un["main_id"]) & (sin_cero(it["id"]) == sin_cero(un["item_id_u"])), "left")
         .select(
            it["main_id"].alias("release_id"), it["main_ocid"].alias("ocid"),
            sin_cero(it["id"]).alias("item_id"),
            it["description"].alias("descripcion_item"),
            normalizar(it["description"]).alias("texto_item"),
            it["quantity"].alias("cantidad"), it["unit_name"].alias("unidad"),
            it["totalValue_amount"].alias("valor_item"), it["totalValue_currency"].alias("moneda_item"),
            sin_cero(it["classification_id"]).alias("cubso_id"),
            it["classification_description"].alias("cubso_desc"),
            sin_cero(un["unspsc_id"]).alias("unspsc_id"), un["unspsc_desc"],
            it["status"].alias("estado_item"), it["statusDetails"].alias("estado_item_detalle"),
         )
         .withColumn("unspsc_segmento", F.substring("unspsc_id", 1, 2))
         .withColumn("unspsc_familia", F.substring("unspsc_id", 1, 4)))
items.repartition(4).write.mode("overwrite").parquet(f"{PQ}/items")

# ── Postores (quienes se presentaron) ─────────────────────────
tt = vigentes(leer("tender_tenderers", COLS_TENDERERS)).cache()
CALIDAD["postores_filas_leidas"] = tt.count()
postores = (tt.select(F.col("main_id").alias("release_id"), F.col("main_ocid").alias("ocid"),
                      ruc_de("id").alias("ruc"), F.col("name").alias("postor"))
              .withColumn("extranjero", F.col("ruc").startswith("L"))
              .dropDuplicates(["release_id", "ruc"]))
postores.repartition(8).write.mode("overwrite").parquet(f"{PQ}/postores")
tt.unpersist()

# ── Adjudicaciones (+ proveedor, estado y monto en soles) ─────
aw = vigentes(leer("awards", COLS_AWARDS))
su = vigentes(leer("awards_suppliers", COLS_SUPPLIERS)).dropDuplicates(["main_id", "awards_id", "id"])
est_adj = (resumen_estado(vigentes(leer("awards_items", COLS_AWARD_ITEMS)), ["main_id", "awards_id"], "statusDetails")
           .select(F.col("main_id").alias("_ea_m"), F.col("awards_id").alias("_ea_a"), F.col("estado").alias("estado_adj")))
fx_a = (vigentes(leer("awards_value_exchangeRates", COLS_FX_AWARDS)).dropDuplicates(["main_id", "awards_id"])
        .select(F.col("main_id").alias("_fa_m"), F.col("awards_id").alias("_fa_a"), F.col("rate").alias("_tc")))
adjud_base = (aw.join(su, (aw["main_id"] == su["main_id"]) & (aw["id"] == su["awards_id"]), "left")
                .join(est_adj, (aw["main_id"] == F.col("_ea_m")) & (aw["id"] == F.col("_ea_a")), "left")
                .join(fx_a, (aw["main_id"] == F.col("_fa_m")) & (aw["id"] == F.col("_fa_a")), "left")
                .select(aw["main_id"].alias("release_id"), aw["main_ocid"].alias("ocid"),
                        aw["id"].alias("adj_id"), aw["date"].alias("fecha_adj"),
                        aw["value_amount"].alias("monto"), aw["value_currency"].alias("moneda"),
                        F.when(aw["value_currency"] == "PEN", F.lit(1.0)).otherwise(F.col("_tc")).alias("tipo_cambio"),
                        ruc_de(su["id"]).alias("proveedor_ruc"), su["name"].alias("proveedor"),
                        F.coalesce(F.col("estado_adj"), F.lit("sin_dato")).alias("estado_adj"))
                .withColumn("monto_pen", F.col("monto") * F.col("tipo_cambio"))
                .withColumn("proveedor_extranjero", F.col("proveedor_ruc").startswith("L"))
                .cache())

# ── Contratos (+ estado, tipo de cambio y monto en soles) ─────
co = vigentes(leer("contracts", COLS_CONTRACTS)).cache()
CALIDAD["contratos_sin_estado"] = co.filter(F.col("status").isNull()).count()   # el estado de la fuente viene vacío
ci = vigentes(leer("contracts_items", COLS_CONTRACT_ITEMS)).withColumn("_s", F.upper(F.trim(F.col("statusDetails"))))
est_con = (resumen_estado(ci, ["main_id", "contracts_id"], "_s")
           .select(F.col("main_id").alias("_ec_m"), F.col("contracts_id").alias("_ec_c"), F.col("estado").alias("estado_contrato")))
fx_c = (vigentes(leer("contracts_value_exchangeRates", COLS_FX_CONTRACTS)).dropDuplicates(["main_id", "contracts_id"])
        .select(F.col("main_id").alias("_fc_m"), F.col("contracts_id").alias("_fc_c"), F.col("rate").alias("_tcc")))
tasas_adj = (adjud_base.select(F.col("release_id").alias("_r"), F.col("adj_id").alias("_a"),
                               F.col("moneda").alias("_ma"), F.col("tipo_cambio").alias("_tca")).distinct())
contratos = (co.join(est_con, (co["main_id"] == F.col("_ec_m")) & (co["id"] == F.col("_ec_c")), "left")
               .join(fx_c, (co["main_id"] == F.col("_fc_m")) & (co["id"] == F.col("_fc_c")), "left")
               .join(tasas_adj, (co["main_id"] == F.col("_r")) & (co["awardID"] == F.col("_a")), "left")
               .select(co["main_id"].alias("release_id"), co["main_ocid"].alias("ocid"),
                       co["id"].alias("contrato_id"), co["awardID"].alias("adj_id"),
                       co["dateSigned"].alias("fecha_firma"),
                       co["value_amount"].alias("monto"), co["value_currency"].alias("moneda"),
                       # tipo de cambio: 1 si ya está en soles; si no, el del propio contrato; si no, el de su adjudicación (misma moneda)
                       F.when(co["value_currency"] == "PEN", F.lit(1.0))
                        .when(F.col("_tcc").isNotNull(), F.col("_tcc"))
                        .when(F.col("_ma") == co["value_currency"], F.col("_tca"))
                        .alias("tipo_cambio"),
                       co["implementation_finalValue_amount"].alias("monto_final"),
                       co["implementation_finalValue_currency"].alias("moneda_final"),
                       co["period_startDate"].alias("inicio"), co["period_endDate"].alias("fin"),
                       co["period_durationInDays"].alias("duracion_dias"),
                       F.coalesce(F.col("estado_contrato"), F.lit("sin_dato")).alias("estado_contrato"),
                       F.col("_a").isNotNull().alias("adjudicacion_vinculada"))
               .withColumn("monto_pen", F.col("monto") * F.col("tipo_cambio"))
               .select("release_id", "ocid", "contrato_id", "adj_id", "fecha_firma", "monto", "moneda", "tipo_cambio",
                       "monto_pen", "monto_final", "moneda_final", "inicio", "fin", "duracion_dias",
                       "estado_contrato", "adjudicacion_vinculada"))
contratos.repartition(2).write.mode("overwrite").parquet(f"{PQ}/contratos")

# adjudicaciones: marcar las que ya tienen contrato
claves_con = (spark.read.parquet(f"{PQ}/contratos").select(F.col("release_id").alias("_r2"), F.col("adj_id").alias("_a2"))
              .distinct().withColumn("_c", F.lit(True)))
adjud = (adjud_base.join(claves_con, (adjud_base["release_id"] == F.col("_r2")) & (adjud_base["adj_id"] == F.col("_a2")), "left")
                   .withColumn("tiene_contrato", F.coalesce(F.col("_c"), F.lit(False)))
                   .select("release_id", "ocid", "adj_id", "fecha_adj", "monto", "moneda", "tipo_cambio", "monto_pen",
                           "proveedor_ruc", "proveedor", "proveedor_extranjero", "estado_adj", "tiene_contrato"))
adjud.repartition(2).write.mode("overwrite").parquet(f"{PQ}/adjudicaciones")
adjud_base.unpersist(); co.unpersist()

# ── Entidades: ubicación (tabla parties, solo filas "buyer") y tipo ──
rel = F.broadcast(base.select(F.col("release_id").alias("_rel"), F.col("entidad_id").alias("_eid"),
                              F.col("fecha_release").alias("_frel")))
pa = (vigentes(leer("parties", COLS_PARTIES))
      .filter(F.col("roles").contains("buyer") & F.col("address_department").isNotNull())
      .join(rel, F.col("main_id") == F.col("_rel")))
# Si una entidad cambió de departamento, se queda con el más reciente (según la fecha de la versión del proceso)
ubic = (pa.groupBy(F.col("_eid").alias("entidad_id"))
          .agg(F.max_by("address_department", "_frel").alias("entidad_departamento"),
               F.max_by("address_region", "_frel").alias("entidad_provincia"),     # en la fuente se llama "region" pero es la provincia
               F.max_by("address_locality", "_frel").alias("entidad_distrito"),
               F.countDistinct("address_department").alias("n_departamentos")))
ent_nom = (base.groupBy("entidad_id")
               .agg(F.max_by("entidad", "fecha_release").alias("entidad"),
                    F.max_by("entidad_tipo", "fecha_release").alias("entidad_tipo"),
                    F.count("*").alias("n_procesos")))
entidades = (ent_nom.join(ubic, "entidad_id", "left")
                    .select("entidad_id", "entidad", "entidad_tipo", "entidad_departamento",
                            "entidad_provincia", "entidad_distrito", "n_departamentos", "n_procesos"))
entidades.coalesce(1).write.mode("overwrite").parquet(f"{PQ}/entidades")
print("Tablas hijas escritas.")

Tablas hijas escritas.


## 6. Tabla final `procesos` (con agregados por proceso)

Une `procesos_base` con conteos desde las tablas hijas, el **estado del proceso** (el estado único de sus ítems, o `MIXTO`)
y la **ubicación de la entidad**. Se particiona por `anio_pub`.

In [ ]:
items_p = spark.read.parquet(f"{PQ}/items")
post_p  = spark.read.parquet(f"{PQ}/postores")
adj_p   = spark.read.parquet(f"{PQ}/adjudicaciones")
con_p   = spark.read.parquet(f"{PQ}/contratos")
ent_p   = spark.read.parquet(f"{PQ}/entidades")

# Estado del proceso: el estado único de sus ítems, o MIXTO si difieren
a_items = (items_p.groupBy("release_id")
           .agg(F.count("*").alias("n_items"),
                F.countDistinct("unspsc_segmento").alias("n_segmentos_unspsc"),
                F.countDistinct("estado_item_detalle").alias("_nd"), F.max("estado_item_detalle").alias("_ed"),
                F.countDistinct("estado_item").alias("_ng"), F.max("estado_item").alias("_eg"))
           .withColumn("estado_proceso", F.when(F.col("_nd") == 1, F.col("_ed")).when(F.col("_nd") > 1, F.lit("MIXTO")))
           .withColumn("estado_proceso_grueso", F.when(F.col("_ng") == 1, F.col("_eg")).when(F.col("_ng") > 1, F.lit("MIXTO")))
           .drop("_nd", "_ed", "_ng", "_eg"))
a_post  = post_p.groupBy("release_id").agg(F.count("*").alias("n_postores"))
a_adj   = adj_p.groupBy("release_id").agg(
            F.count("*").alias("n_adjudicaciones"),
            F.countDistinct("proveedor_ruc").alias("n_proveedores_adj"),
            F.sum("monto_pen").alias("monto_adj_pen"),
            F.sum(F.when(F.col("moneda") != "PEN", 1).otherwise(0)).alias("n_adj_otra_moneda"),
            F.sum(F.when(F.col("monto_pen").isNull(), 1).otherwise(0)).alias("n_adj_sin_monto_pen"))
a_con   = con_p.groupBy("release_id").agg(
            F.count("*").alias("n_contratos"),
            F.sum((~F.col("adjudicacion_vinculada")).cast("int")).alias("n_contratos_sin_adj"),
            F.sum((F.col("estado_contrato") == "RESUELTO").cast("int")).alias("n_contratos_resueltos"))
ubi     = ent_p.select("entidad_id", "entidad_departamento", "entidad_provincia", "entidad_distrito")

procesos = base
for a in (a_items, a_post, a_adj, a_con):
    procesos = procesos.join(a, "release_id", "left")
procesos = procesos.join(ubi, "entidad_id", "left")
for c in ["n_items", "n_segmentos_unspsc", "n_postores", "n_adjudicaciones", "n_proveedores_adj",
          "n_adj_otra_moneda", "n_adj_sin_monto_pen", "n_contratos", "n_contratos_sin_adj", "n_contratos_resueltos"]:
    procesos = procesos.withColumn(c, F.coalesce(F.col(c), F.lit(0)))
for c in ["estado_proceso", "estado_proceso_grueso"]:
    procesos = procesos.withColumn(c, F.coalesce(F.col(c), F.lit("sin_dato")))
procesos = (procesos.withColumn("tiene_adjudicacion", F.col("n_adjudicaciones") > 0)
                    .withColumn("tiene_contrato", F.col("n_contratos") > 0))

procesos.write.mode("overwrite").partitionBy("anio_pub").parquet(f"{PQ}/procesos")
base.unpersist()
shutil.rmtree(f"{PQ}/_procesos_base", ignore_errors=True)
print("procesos escrito.")

procesos escrito.


## 7. Reporte de calidad (para el informe)

In [ ]:
procesos = spark.read.parquet(f"{PQ}/procesos")
items_p  = spark.read.parquet(f"{PQ}/items")
post_p   = spark.read.parquet(f"{PQ}/postores")
adj_p    = spark.read.parquet(f"{PQ}/adjudicaciones")
con_p    = spark.read.parquet(f"{PQ}/contratos")
ent_p    = spark.read.parquet(f"{PQ}/entidades")

CALIDAD["postores_duplicados_eliminados"] = CALIDAD["postores_filas_leidas"] - post_p.count()
CALIDAD["procesos_por_anio_pub"] = {int(r["anio_pub"]): r["n"] for r in procesos.groupBy("anio_pub").agg(F.count("*").alias("n")).orderBy("anio_pub").collect()}
CALIDAD["procesos_metodo_sin_dato"] = procesos.filter("metodo = 'sin_dato'").count()
CALIDAD["procesos_n_postores_declarado_nulo"] = procesos.filter(F.col("n_postores_declarado").isNull()).count()
CALIDAD["procesos_sin_adjudicacion"] = procesos.filter(~F.col("tiene_adjudicacion")).count()
CALIDAD["items_total"] = items_p.count()
CALIDAD["items_sin_cubso"] = items_p.filter(F.col("cubso_id").isNull()).count()
CALIDAD["items_sin_unspsc"] = items_p.filter(F.col("unspsc_id").isNull()).count()
CALIDAD["adjudicaciones_total"] = adj_p.count()
CALIDAD["adjudicaciones_sin_proveedor"] = adj_p.filter(F.col("proveedor_ruc").isNull()).count()
CALIDAD["adjudicaciones_sin_contrato"] = adj_p.filter(~F.col("tiene_contrato")).count()
CALIDAD["proveedores_extranjeros_en_adj"] = adj_p.filter("proveedor_extranjero").count()
CALIDAD["contratos_total"] = con_p.count()
CALIDAD["contratos_sin_adjudicacion_vinculada"] = con_p.filter(~F.col("adjudicacion_vinculada")).count()

def conteo(df, col):
    return {str(r[col]): r["n"] for r in df.groupBy(col).agg(F.count("*").alias("n")).orderBy(F.desc("n")).collect()}

# Columnas nuevas: ubicación, estados y tipo de cambio
CALIDAD["entidades_total"] = ent_p.count()
CALIDAD["entidades_sin_departamento"] = ent_p.filter(F.col("entidad_departamento").isNull()).count()
CALIDAD["entidades_con_cambio_departamento"] = ent_p.filter(F.col("n_departamentos") > 1).count()
CALIDAD["procesos_sin_departamento"] = procesos.filter(F.col("entidad_departamento").isNull()).count()
CALIDAD["procesos_por_estado"] = conteo(procesos, "estado_proceso")
CALIDAD["procesos_sin_adjudicacion_por_estado"] = conteo(procesos.filter(~F.col("tiene_adjudicacion")), "estado_proceso")
CALIDAD["procesos_por_tipo_entidad"] = conteo(procesos, "entidad_tipo")
CALIDAD["adjudicaciones_por_estado"] = conteo(adj_p, "estado_adj")
CALIDAD["adjudicaciones_moneda_extranjera"] = adj_p.filter(F.col("moneda") != "PEN").count()
CALIDAD["adjudicaciones_sin_tipo_cambio"] = adj_p.filter(F.col("monto_pen").isNull()).count()
CALIDAD["contratos_por_estado"] = conteo(con_p, "estado_contrato")
CALIDAD["contratos_moneda_extranjera"] = con_p.filter(F.col("moneda") != "PEN").count()
CALIDAD["contratos_sin_tipo_cambio"] = con_p.filter(F.col("monto_pen").isNull()).count()

with open(f"{SALIDA_DIR}/reporte_calidad.json", "w", encoding="utf-8") as f:
    json.dump(CALIDAD, f, ensure_ascii=False, indent=2, default=str)

pd.Series({k: v for k, v in CALIDAD.items() if not isinstance(v, dict)}, name="valor").to_frame()

,valor
main_filas_leidas,357729
main_ocid_duplicados_eliminados,0
main_fuera_de_periodo,147341
main_mes_parcial_excluido,648
mes_parcial_excluido,2026-09
procesos_final,209740
procesos_valor_cero,21718
procesos_info_protegida,22483
procesos_valor_cero_y_protegido,16851
fecha_corte,2026-09-03 10:51:02.475118


In [ ]:
# % de nulos por columna en procesos (Spark, una sola pasada)
n = procesos.count()
nulos = procesos.select([F.round(100 * F.avg(F.col(c).isNull().cast("int")), 1).alias(c) for c in procesos.columns]).first().asDict()
pd.Series(nulos, name="% nulo").sort_values(ascending=False).to_frame().query("`% nulo` > 0")

,% nulo
proyecto_id,59.7
monto_adj_pen,23.8
valor_pen_valido,10.4
n_postores_declarado,10.2
fecha_fin_conv,6.0
fecha_inicio_conv,6.0


## 8. Esquema final y validaciones

> **Ojo al explorar este notebook:** las variables de la sección 5 (`items`, `postores`, `adjud`, `contratos`, `entidades`) son
> consultas temporales (dependen de una carpeta que se borra al final de la sección 6) y **ya no funcionan**.
> Para consultar los datos finales usa las leídas desde Parquet: `procesos`, `items_p`, `post_p`, `adj_p`, `con_p` y `ent_p`.

In [ ]:
for t in ["procesos", "items", "postores", "adjudicaciones", "contratos", "entidades"]:
    df = spark.read.parquet(f"{PQ}/{t}")
    print(f"\n=== {t}  |  filas: {df.count():,}  |  particiones al leer: {df.rdd.getNumPartitions()}")
    df.printSchema()


=== procesos  |  filas: 209,740  |  particiones al leer: 3
root
 |-- entidad_id: string (nullable = true)
 |-- release_id: string (nullable = true)
 |-- ocid: string (nullable = true)
 |-- fecha_release: timestamp_ntz (nullable = true)
 |-- proceso_id: string (nullable = true)
 |-- entidad: string (nullable = true)
 |-- titulo: string (nullable = true)
 |-- descripcion: string (nullable = true)
 |-- metodo: string (nullable = true)
 |-- procedimiento: string (nullable = true)
 |-- categoria: string (nullable = true)
 |-- fecha_pub: timestamp_ntz (nullable = true)
 |-- fecha_inicio_conv: timestamp_ntz (nullable = true)
 |-- fecha_fin_conv: timestamp_ntz (nullable = true)
 |-- valor_orig: double (nullable = true)
 |-- moneda: string (nullable = true)
 |-- valor_pen: double (nullable = true)
 |-- n_postores_declarado: integer (nullable = true)
 |-- proyecto_id: string (nullable = true)
 |-- info_protegida: boolean (nullable = true)
 |-- archivo_origen: integer (nullable = true)
 |-- mes_

In [ ]:
# Integridad referencial: ninguna tabla hija debe tener release_id que no exista en procesos
ids_p = procesos.select("release_id")
for nombre, df in [("items", items_p), ("postores", post_p), ("adjudicaciones", adj_p), ("contratos", con_p)]:
    huerfanas = df.join(ids_p, "release_id", "left_anti").count()
    print(f"{nombre:15s} filas huérfanas: {huerfanas}")

# ocid único
assert procesos.count() == procesos.select("ocid").distinct().count(), "ocid duplicado en procesos"
print("ocid único: OK")

items           filas huérfanas: 0
postores        filas huérfanas: 0
adjudicaciones  filas huérfanas: 0
contratos       filas huérfanas: 0
ocid único: OK


In [ ]:

# Columnas nuevas: cobertura y distribución
proc = spark.read.parquet(f"{PQ}/procesos")
ent  = spark.read.parquet(f"{PQ}/entidades")
n = proc.count()

def distrib(df, col, k=12):
    t = df.count()
    d = (df.groupBy(col).count().withColumn("%", F.round(100 * F.col("count") / t, 1))
           .orderBy(F.desc("count")).limit(k).toPandas())
    return d.to_string(index=False)

print("== Ubicación de la entidad ==")
print("entidades:", ent.count(), "| sin departamento:", ent.filter(F.col("entidad_departamento").isNull()).count(),
      "| con cambio de departamento:", ent.filter(F.col("n_departamentos") > 1).count())
print("procesos sin departamento:", proc.filter(F.col("entidad_departamento").isNull()).count(),
      "| departamentos distintos:", proc.select("entidad_departamento").distinct().count())
print(distrib(proc, "entidad_departamento", 8))

print("\n== Tipo de entidad (heurístico) ==");  print(distrib(proc, "entidad_tipo"))
print("\n== Estado del proceso ==");            print(distrib(proc, "estado_proceso"))
print("\n== Procesos SIN adjudicación, por estado ==")
print(distrib(proc.filter(~F.col("tiene_adjudicacion")), "estado_proceso", 8))

adj = spark.read.parquet(f"{PQ}/adjudicaciones"); con = spark.read.parquet(f"{PQ}/contratos")
print("\n== Adjudicaciones sin contrato, por estado de la adjudicación ==")
print(distrib(adj.filter(~F.col("tiene_contrato")), "estado_adj", 6))
print("\n== Estado del contrato ==");            print(distrib(con, "estado_contrato", 6))

print("\n== Tipo de cambio (monto en soles) ==")
for nombre, df in [("adjudicaciones", adj), ("contratos", con)]:
    ext = df.filter(F.col("moneda") != "PEN"); t = ext.count()
    ok = ext.filter(F.col("monto_pen").isNotNull()).count()
    print(f"{nombre:15s} en otra moneda: {t:,} | con monto en soles: {ok:,} ({100*ok/max(t,1):.1f}%)")


== Ubicación de la entidad ==
entidades: 2992 | sin departamento: 0 | con cambio de departamento: 2
procesos sin departamento: 0 | departamentos distintos: 25
entidad_departamento  count    %
                LIMA  62796 29.9
               CUSCO  17847  8.5
              ANCASH  11935  5.7
                PUNO   9970  4.8
               JUNIN   8650  4.1
            AREQUIPA   8458  4.0
            AYACUCHO   7982  3.8
           CAJAMARCA   7802  3.7

== Tipo de entidad (heurístico) ==
     entidad_tipo  count    %
    municipalidad  87155 41.6
             otro  47551 22.7
gobierno_regional  46117 22.0
            salud   9707  4.6
      universidad   8052  3.8
  defensa_policia   6881  3.3
       ministerio   4277  2.0

== Estado del proceso ==
                              estado_proceso  count    %
                                  CONTRATADO 131154 62.5
                                    DESIERTO  25268 12.0
                                   CONVOCADO  13577  6.5
              

In [ ]:
# Vistazo: procesos por mes
(procesos.groupBy("periodo").agg(F.count("*").alias("procesos"))
         .orderBy("periodo").toPandas().tail(14))

,periodo,procesos
18,2025-07,6715
19,2025-08,7478
20,2025-09,7707
21,2025-10,8307
22,2025-11,7079
23,2025-12,8519
24,2026-01,994
25,2026-02,4884
26,2026-03,8289
27,2026-04,7958


In [ ]:
procesos.select("ocid", "entidad", "procedimiento", "categoria", "fecha_pub", "valor_pen_valido",
                "info_protegida", "entidad_departamento", "entidad_tipo", "estado_proceso", "n_postores", "n_items", "tiene_adjudicacion", "texto").limit(5).toPandas()

,ocid,entidad,procedimiento,categoria,fecha_pub,valor_pen_valido,info_protegida,entidad_departamento,entidad_tipo,estado_proceso,n_postores,n_items,tiene_adjudicacion,texto
0,ocds-dgv273-seacev3-1081886,PETROLEOS DEL PERU S.A.,Adjudicación Abreviada,services,2025-01-07 10:36:00,NaN,False,LIMA,otro,CONTRATADO,2,1,True,servicio de recuperacion de residuos de hidroc...
1,ocds-dgv273-seacev3-1082466,PETROLEOS DEL PERU S.A.,Adjudicación Abreviada,services,2025-01-21 17:23:00,NaN,False,LIMA,otro,CONTRATADO,3,1,True,servicio de recoleccion de maleza vegetacion y...
2,ocds-dgv273-seacev3-1083683,MINISTERIO DE JUSTICIA Y DERECHOS HUMANOS,Regímen Especial,works,2025-01-02 11:00:00,5029850.18,False,LIMA,ministerio,CONTRATADO,1,1,True,contratacion para el financiamiento y ejecucio...
3,ocds-dgv273-seacev3-1085253,MUNICIPALIDAD PROVINCIAL DE ICA,Licitación Pública,works,2025-01-02 07:07:00,12997299.89,False,ICA,municipalidad,CONTRATADO,59,1,True,ejecucion de obra denominado mejoramiento y am...
4,ocds-dgv273-seacev3-1085502,MUNICIPALIDAD PROVINCIAL DE CORONEL PORTILLO,Comparación de Precios,goods,2025-01-02 16:23:00,NaN,False,UCAYALI,municipalidad,CONTRATADO,3,1,True,adquisicion de materiales para senales vertica...


In [ ]:
# Procesos por mes (filas) y año de publicación (columnas)
procesos.groupBy("mes_pub").pivot("anio_pub").count().orderBy("mes_pub").show()

# Mezcla de procedimientos alrededor del cambio de ley (22-abr-2025)
(procesos.filter(F.col("periodo").between("2025-03", "2025-07"))
         .groupBy("periodo", "procedimiento").count()
         .orderBy("periodo", F.desc("count")).show(40, False))

+-------+----+----+----+
|mes_pub|2024|2025|2026|
+-------+----+----+----+
|      1|1008|1231| 994|
|      2|4382|4980|4884|
|      3|6340|8199|8289|
|      4|8528|9356|7958|
|      5|8337|3646|7853|
|      6|6941|5880|7377|
|      7|6745|6715|6600|
|      8|6863|7478|6579|
|      9|7164|7707|NULL|
|     10|8346|8307|NULL|
|     11|7030|7079|NULL|
|     12|8425|8519|NULL|
+-------+----+----+----+

+-------+----------------------------------------------------------------------+-----+
|periodo|procedimiento                                                         |count|
+-------+----------------------------------------------------------------------+-----+
|2025-03|Adjudicación Simplificada                                             |4473 |
|2025-03|Subasta Inversa Electrónica                                           |1247 |
|2025-03|Regímen Especial                                                      |695  |
|2025-03|Comparación de Precios                                              

### Nota: estacionalidad y cambio de ley en 2025

**Enero es bajo todos los años** (2024: 1,008 · 2025: 1,231 · 2026: 994). Es el inicio del año fiscal.
No es un error de los datos ni de la limpieza.

**Mayo de 2025 cae a menos de la mitad de lo normal:**

| Mes | 2024 | 2025 | 2026 |
|---|---|---|---|
| Abril | 8,528 | 9,356 | 7,958 |
| Mayo | 8,337 | 3,646 | 7,853 |
| Junio | 6,941 | 5,880 | 7,377 |

**Contexto:** la nueva Ley General de Contrataciones Públicas (Ley 32069) entró en vigencia el 22 de abril de 2025.
El 21 de abril, último día de la ley anterior, se registraron 2,135 convocatorias en un solo día, el máximo
diario de la historia del Seace.
Fuentes:
- https://gestion.pe/economia/seace-nuevo-regimen-de-contrataciones-inicia-con-68-procedimientos-registrados-licitacion-publica-subasta-contratacion-entidades-del-estado-noticia/
- https://www.elperuano.pe/noticia/265279-nueva-ley-de-contrataciones-publicas-entra-en-vigencia-el-22-de-abril-de-2025

**Interpretación (hipótesis, respaldada solo en parte):** una posible explicación es que las entidades adelantaron
convocatorias a abril y que mayo cayó por la transición. Los datos muestran el adelanto solo en algunos procedimientos
(ver abajo). No se corrige ni se excluye este periodo.

**Qué implica:**
- En el EDA temporal hay que explicar este salto, para que no parezca una falla.
- Después de abril 2025 aparecen procedimientos nuevos (por ejemplo, Licitación Pública Abreviada y Concurso Público
  Abreviado). Al mezclar antes y después se combinan dos regímenes.
- La salida de la celda anterior lo muestra. En marzo 2025 no hay Licitación Pública Abreviada ni Concurso Público Abreviado.
  En abril aparecen los primeros (20 y 108 procesos, pocos porque la ley rige desde el 22). En mayo la Licitación Pública
  Abreviada ya es el segundo procedimiento más frecuente (531).
- Entre marzo y abril 2025 el cuadro de los procedimientos antiguos es mixto. Concurso Público (313 → 631) y
  Licitación Pública (341 → 617) suben mucho, pero Subasta Inversa Electrónica baja (1,247 → 920) y Régimen Especial
  cae de 695 a 56. El total de abril sube solo 14%. Por eso el adelanto de convocatorias es una hipótesis parcial.
  Después de abril, "Concurso Público" deja de aparecer.
  

## 9. Qué se entrega

Carpeta `parquet/` dentro de la salida (`DRIVE_DIR`). Las tablas se enlazan por `release_id`. En `procesos`, `ocid` también es único.
`entidades` se une a `procesos` por `entidad_id`.
Los pasos para usarlas están en la **sección 10**, y el diccionario de columnas en la **sección 11**.

| Tabla | Filas | Una fila por | Uso sugerido |
|---|---|---|---|
| `procesos` | 209,740 | proceso de contratación | Todos (EDA, flujo por `fecha_pub`, texto para similitud) |
| `items` | 243,699 | ítem convocado (con CUBSO, UNSPSC y estado) | C (canastas) y A (representación) |
| `postores` | 2,493,612 | postor por proceso | B (Bloom, CMS) |
| `adjudicaciones` | 174,908 | adjudicación (+ proveedor, estado y monto en soles) | B y C |
| `contratos` | 153,272 | contrato (+ estado y monto en soles) | B y C |
| `entidades` | 2,992 | entidad contratante (con ubicación y tipo) | B (EDA, mapa) y C (canastas) |

**Fechas:** todas están en **hora de Lima (UTC−5), sin zona horaria**. No se convierten.

**`procesos` está particionada por `anio_pub`** (subcarpetas `anio_pub=2024`, `anio_pub=2025`, ...).
Lee siempre la carpeta `procesos` completa. Si lees una subcarpeta, se pierde la columna `anio_pub`.

**Para agregar años:** cambia `ANIOS` (y `FECHA_PUB_MIN` si hace falta) y ejecuta todo de nuevo.
Los años que ya estén descomprimidos en `/content/raw` se reutilizan.
Esa carpeta se borra al cerrar Colab. Si pasa, se vuelve a extraer desde los zips de `ZIPS_DIR`.

**Antes de volver a ejecutar el notebook:** la carpeta `parquet/` se **borra y se reescribe completa**.
Avisa antes a B y C, porque si están leyendo en ese momento, les fallará.

## 10. Pasos para B y C

> El diccionario de cada tabla, las decisiones tomadas y los ejemplos de uniones están en la **sección 11**.

### Antes de empezar
1. **Pide acceso** a la carpeta `parquet` que comparte A (con el link de Drive).
2. **Si trabajas en Colab:** en Drive, clic derecho sobre la carpeta compartida → *Organizar* → *Agregar acceso directo* → elige *Mi unidad*.
   Sin esto, tu Colab no la ve.
3. **Si trabajas en tu computadora:** descarga la carpeta `parquet` completa desde Drive.
4. **No necesitas los CSV ni los zips.** Solo los Parquet.

### Leer los datos (Spark)
Copia este bloque en una celda de código de **tu propio notebook**. No hace falta ejecutar el notebook de A.
```python
# ── Solo si trabajas en Colab ──
!pip install -q pyspark==3.5.3
from google.colab import drive
drive.mount("/content/drive")
# (Si trabajas en tu computadora, sáltate estas 3 líneas)
# ────────────────────────────────

from pyspark.sql import SparkSession, functions as F
spark = SparkSession.builder.master("local[*]").config("spark.driver.memory", "8g").getOrCreate()
# NO cambies spark.sql.session.timeZone: las fechas ya están en hora de Lima.

RUTA = "/content/drive/MyDrive/parquet"      # ajusta a donde quedó la carpeta
procesos       = spark.read.parquet(f"{RUTA}/procesos")
items          = spark.read.parquet(f"{RUTA}/items")
postores       = spark.read.parquet(f"{RUTA}/postores")
adjudicaciones = spark.read.parquet(f"{RUTA}/adjudicaciones")
contratos      = spark.read.parquet(f"{RUTA}/contratos")
entidades      = spark.read.parquet(f"{RUTA}/entidades")
```
Con pandas también funciona: `pd.read_parquet(f"{RUTA}/procesos")`. Usa Spark 3.4 o más reciente.

### Reglas para no equivocarse
- **Fechas:** ya están en hora de Lima. **No uses** `tz_convert` ni cambies la zona horaria de Spark. Si lo haces, se corren 5 horas.
- **Agrupar por mes:** usa `periodo` o `mes_pub`.
- **Periodo:** los datos van de **2024-01 a 2026-08**. El último mes incompleto (septiembre 2026, solo 3 días) **ya viene excluido**.
  No hay que filtrar nada. El rango también aparece como `periodo_analisis` en la sección 7.
- **Salto en 2025:** abril 2025 sube y mayo 2025 cae a menos de la mitad. Coincide con el cambio de ley (22-abr-2025). No es un error de datos. Ver la nota de la sección 8.
- **Flujo (Parte IV, B):** ordena por `fecha_pub`.
- **Tipo de procedimiento:** usa `procedimiento`, no `metodo`. `metodo` está vacío en cerca del 30% (viene `sin_dato`).
- **Cómo terminó un proceso:** usa `estado_proceso` (CONTRATADO, DESIERTO, NULO, CANCELADO, CONVOCADO...). **Un proceso sin adjudicación no es un fracaso:**
  el 26% de ellos sigue abierto (`CONVOCADO`). Para medir tasas de desierto o de adjudicación, excluye `CONVOCADO` y ten en cuenta que los procesos recientes están más abiertos.
- **Montos de procesos:** usa `valor_pen_valido`. Los `0` se tratan como valor no disponible. El 78% son procesos con información protegida por ley (`info_protegida = true`).
- **Montos de adjudicaciones y contratos:** usa `monto_pen`, que ya está en soles (con el tipo de cambio de la fuente). Queda nulo en 95 adjudicaciones y 76 contratos
  en moneda extranjera que no traen tipo de cambio. El `monto` original sigue en su moneda (`moneda`).
- **Contratos:** usa `estado_contrato` (CONTRATADO, RESUELTO, NULO). Se deduce de los ítems y el 15% queda `sin_dato`.
- **Ubicación:** `entidad_departamento`, `entidad_provincia` y `entidad_distrito` son la **sede de la entidad**, no el lugar donde se ejecuta la obra o el servicio.
  Solo existe para la entidad, no para postores ni proveedores.
- **Tipo de entidad:** `entidad_tipo` es heurístico (se deduce del nombre). El 23% queda como `otro`.
- **Unir tablas:** usa `release_id`. Varias tablas comparten columnas (`ocid`, `moneda`, `monto`, `tipo_cambio`, `monto_pen`, `tiene_contrato`):
  elimínalas o renómbralas antes de unir, o quedarán ambiguas. Ver ejemplos en la sección 11.
  Una adjudicación puede tener más de un contrato, así que al unirlas su fila se repite.
- **Texto para similitud:** la columna `texto` de `procesos` ya viene normalizada (minúsculas, sin tildes).

### Tablas sugeridas para cada uno
| Persona | Tablas principales |
|---|---|
| **B** (EDA, Bloom, CMS, DGIM, web) | `procesos`, `postores`, `adjudicaciones`, `contratos`, `entidades` |
| **C** (agregaciones, Apriori, FP-Growth) | `procesos`, `items`, `adjudicaciones`, `contratos`, `entidades` |

Es solo una sugerencia. Cada uno puede usar las tablas que necesite.

### Si algo falta o se ve raro
- Si necesitas una columna que no está, avisa a A. No limpies los CSV por tu cuenta, porque los resultados ya no coincidirían.
- Todos deben trabajar sobre los mismos Parquet. Si A los vuelve a generar, avisará antes.
- Los conteos de la limpieza (filas leídas, duplicados, nulos, etc.) están en la sección 7 de este notebook.

## 11. Contexto, decisiones y diccionario de datos

> Esta sección documenta cómo se generaron los datos. **No hace falta volver a ejecutar este notebook**
> (necesita los zips originales). Para trabajar, basta con leer los Parquet de la carpeta `parquet/`.

### Contexto
Proyecto Integrador del curso Minería de Datos Masivos. Dataset: contrataciones públicas del Perú (SEACE, formato OCDS, OECE).
Técnicas del proyecto: similitud (MinHash/LSH), vecinos más cercanos (ANN), minería de flujos (Bloom, Count-Min, DGIM) y reglas de asociación.

### Fuente y licencia
- **Fuente:** Organismo Especializado para las Contrataciones Públicas Eficientes (OECE, antes OSCE), obtenido vía Open Contracting Partnership:
  https://data.open-contracting.org/es/publication/135
- **Licencia:** Creative Commons Atribución 4.0 Internacional (CC BY 4.0): https://creativecommons.org/licenses/by/4.0/
  Exige dar crédito, enlazar la licencia e **indicar si se hicieron cambios**. Estos datos fueron filtrados, limpiados y transformados.
- **Fecha de los datos:** la fuente indica que se obtuvo por última vez el 4 de septiembre de 2026. La última actualización
  incluida en estos datos es del 3 de septiembre de 2026.

### Cobertura y limitaciones
- **No incluye** las contrataciones sin procedimiento de selección (por ejemplo, las menores a 3, 8 o 9 UIT). Los resultados
  describen los procedimientos de selección, no toda la compra pública.
- **Fechas:** `fecha_pub` coincide con el inicio de la convocatoria el mismo día en el 94.7% de los procesos que tienen fecha de inicio.
  En el 0.9% hay más de 90 días de diferencia.
- **Los estados son a la fecha de corte** (3 de septiembre de 2026). Los procesos recientes están más "abiertos" que los antiguos.
- **Estados que no cuadran:** 4,889 procesos (2.3%) tienen adjudicación, pero sus ítems figuran como desiertos, nulos, cancelados o convocados.
- **Problemas de calidad que declara la fuente**, y lo que se encontró en estos datos:

| Problema declarado | En estos datos |
|---|---|
| Postores con identificadores duplicados | 0 duplicados |
| Contratos sin estado | El estado oficial viene vacío en el 100%. Se deduce de los ítems del contrato (ver decisión 12) |
| Adjudicaciones sin contrato vinculado | 16% de las adjudicaciones. El 92% están en "consentido" o "adjudicado" (contrato aún no firmado) |
| Contratos sin adjudicación vinculada | 1.1% de los contratos |

### Decisiones tomadas
1. **Fuente:** OCDS Perú, CSV por año. El año del archivo es el año de la **última actualización**, no el de publicación.
2. **Periodo:** procesos **publicados entre 2024-01 y 2026-08**. Volumen suficiente (millones de filas) y datos recientes.
3. **No se usa el Padrón SUNAT.** El dataset OCDS cumple los requisitos del proyecto por sí solo.
4. **Septiembre 2026 se excluye:** el corte es el 3 de septiembre y ese mes está incompleto.
5. **Duplicados:** si un `ocid` aparece en más de un año, se deja la versión más reciente (no se encontraron).
6. **Fechas:** hora de Lima (UTC−5), sin zona horaria. No convertir.
7. **Monto en 0:** se trata como valor no disponible, no como monto real. Se marca en `valor_cero` y no entra en `valor_pen_valido`.
   - El 78% de los ceros (16,851 de 21,718) son procesos con información protegida por ley (`info_protegida`).
   - De los 4,867 ceros restantes, 1,922 (39%) son de Petroperú. El 86% de los procesos de esa entidad vienen con monto 0 sin esa protección. La causa no está determinada.
   - El resto se reparte entre entidades como la Fuerza Aérea, el Ejército y el JNE.
8. **Se usa `procedimiento` y no `metodo`:** `metodo` está vacío en cerca del 30% (`sin_dato`).
   El 99.6% de esos vacíos son procedimientos nuevos (desde abril 2025). El resto, 0.4%, son sobre todo variantes de
   "Adjudicación Simplificada" de 2024 con norma especial.
9. **Cambio de ley el 22-abr-2025** (Ley 32069): abril 2025 sube y mayo 2025 cae a menos de la mitad.
   Los procedimientos nuevos (Licitación Pública Abreviada, Concurso Público Abreviado, etc.) aparecen desde 2025-04.
   "Concurso Público" y "Selección de Consultores Individuales" terminan ese mes. Otros, como "Adjudicación Simplificada", continúan.
   Ver la nota de la sección 8.
10. **Tablas OCDS no usadas:** documentos (`tender_documents`, `contracts_documents`), `parties_additionalIdentifiers`, `sources`,
    clasificaciones adicionales de adjudicaciones y contratos, y los tipos de cambio de ítems. De `parties` solo se usan las filas de la
    entidad contratante (departamento, provincia y distrito). `awards_items` y `contracts_items` se usan solo para sacar estados.
11. **Ubicación de la entidad:** departamento, provincia y distrito de la **sede** de la entidad contratante, tomados de `parties` (filas "buyer").
    Cobertura del 100% (2,992 entidades, 25 departamentos: los 24 más el Callao). Si una entidad cambió de departamento
    (2 casos, de Callao a Lima), se usa el más reciente. En la fuente, el campo `region` es en realidad la provincia y `locality` el distrito.
12. **Estados:** el estado de un proceso es el estado único de sus ítems (`MIXTO` si difieren, el 3.2%). El estado de una adjudicación sale de sus
    ítems adjudicados, y el de un contrato de sus ítems contratados (la fuente deja vacío el estado del propio contrato).
    Las columnas `estado` y `estado_detalle` de `contratos` (vacías) se reemplazaron por `estado_contrato`.
13. **Montos en soles:** `monto_pen` = monto × tipo de cambio de la fuente (1 si ya está en soles). El tipo de cambio de un contrato es el suyo o,
    si no lo trae, el de su adjudicación (misma moneda). Cobertura: 97.5% de las adjudicaciones y 98.0% de los contratos en moneda extranjera.
    Se usan las tasas tal como las publica la fuente; 2 de 3,160 tasas de dólar son atípicas (cerca de 2.7).
14. **Tipo de entidad:** `entidad_tipo` se deduce de palabras del nombre. Es **heurístico**: no viene en la fuente y el 23% queda como `otro`
    (por ejemplo Petroperú, OEFA, SUNAT).

### Tablas de salida (Parquet)
| Tabla | Filas | Una fila por |
|---|---|---|
| `procesos` | 209,740 | proceso de contratación |
| `items` | 243,699 | ítem convocado (promedio 1.2 por proceso) |
| `postores` | 2,493,612 | postor por proceso (promedio 12 por proceso) |
| `adjudicaciones` | 174,908 | adjudicación con su proveedor |
| `contratos` | 153,272 | contrato |
| `entidades` | 2,992 | entidad contratante |

### De qué tablas de la fuente sale cada cosa
La fuente (OCDS) tiene 21 tablas. Se leen **12** y se generan las 6 tablas de salida de arriba.
Tablas como `parties`, `awards_items` o `contracts_items` no se entregan por separado: se usan para construir columnas.

| Tabla de la fuente | Se usa para | Dónde queda |
|---|---|---|
| `main` | Datos básicos del proceso | `procesos` |
| `tender_items` y `tender_items_additionalClassifications` | Ítems, clasificaciones CUBSO y UNSPSC, y estado de los ítems | `items`, y `estado_proceso` en `procesos` |
| `tender_tenderers` | Postores | `postores` |
| `awards` y `awards_suppliers` | Adjudicaciones y su proveedor | `adjudicaciones` |
| `awards_items` | Estado de la adjudicación | `estado_adj` en `adjudicaciones` |
| `awards_value_exchangeRates` | Tipo de cambio de las adjudicaciones | `tipo_cambio` y `monto_pen` en `adjudicaciones` |
| `contracts` | Contratos | `contratos` |
| `contracts_items` | Estado del contrato | `estado_contrato` en `contratos` |
| `contracts_value_exchangeRates` | Tipo de cambio de los contratos | `tipo_cambio` y `monto_pen` en `contratos` |
| `parties` (solo las filas de la entidad contratante) | Ubicación de la entidad | `entidades`, y `entidad_departamento`, `entidad_provincia` y `entidad_distrito` en `procesos` |

**Sobre `parties`:** tiene 2.7 millones de filas. Las de postores (2,493,612) son las mismas que la tabla `postores`, y las de la
entidad contratante (209,743) son una por proceso. Solo las de la entidad traen ubicación. Los postores y proveedores no la traen,
y su nombre y RUC ya están en `postores` y `adjudicaciones`.

**No se usan (9 tablas):** `tender_documents`, `contracts_documents`, `parties_additionalIdentifiers`, `sources`,
`awards_items_additionalClassifications`, `contracts_items_additionalClassifications`, `awards_items_totalValue_exchangeRates`,
`contracts_items_totalValue_exchangeRates` y `tender_items_totalValue_exchangeRates`.

### Cómo unir las tablas
- `items`, `postores`, `adjudicaciones` y `contratos` se unen a `procesos` por `release_id`.
- En `procesos`, `ocid` también es único y existe en todas las tablas.
- `adjudicaciones` y `contratos` se unen entre sí por `release_id` + `adj_id`.
- `entidades` se une a `procesos` por `entidad_id`. Úsala para análisis por entidad (por ejemplo, contar entidades por departamento).
- **Cuidado con las columnas repetidas.** Varias tablas comparten nombres (`ocid`, `moneda`, `monto`, `tipo_cambio`, `monto_pen`, `tiene_contrato`).
  Elimínalas o renómbralas antes de unir, para que no queden ambiguas:
  ```python
  procesos.join(items.drop("ocid"), "release_id")
  procesos.join(postores.drop("ocid"), "release_id")
  procesos.join(adjudicaciones.drop("ocid", "moneda", "tiene_contrato"), "release_id")
  procesos.join(contratos.drop("ocid", "moneda"), "release_id")

  # adjudicaciones con su contrato (conserva ambos montos)
  adjudicaciones.join(
      contratos.drop("ocid")
               .withColumnRenamed("monto", "monto_contrato").withColumnRenamed("moneda", "moneda_contrato")
               .withColumnRenamed("tipo_cambio", "tipo_cambio_contrato").withColumnRenamed("monto_pen", "monto_pen_contrato"),
      ["release_id", "adj_id"], "left")
  ```
- **Una adjudicación puede tener más de un contrato.** Al unirlas, su fila se repite. Cuenta con `countDistinct("adj_id")`, no con `count`.

### Estados: valores y significado
El significado sale de los nombres y de cómo se comportan los datos. La fuente no los documenta, así que es una **interpretación**.

| `estado_proceso` | Procesos | Qué indica |
|---|---|---|
| `CONTRATADO` | 62.5% | Con contrato |
| `DESIERTO` | 12.0% | Declarado desierto. Es el 48% de los procesos sin adjudicación |
| `CONVOCADO` | 6.5% | Aún abierto a la fecha de corte. Es el 26% de los procesos sin adjudicación |
| `NULO` | 5.7% | Anulado. Es el 18% de los procesos sin adjudicación |
| `CONSENTIDO` | 5.1% | Adjudicación firme, contrato aún no firmado |
| `MIXTO` | 3.2% | Sus ítems tienen estados distintos |
| `ADJUDICADO` | 1.6% | Adjudicado, contrato aún no firmado |
| `RETROTRAIDO_POR_RESOLUCION` | 1.5% | Retrotraído por resolución |
| `CANCELADO` | 1.3% | Cancelado |
| Otros | 0.5% | `PENDIENTE_DE_REGISTRO_DE_EFECTO`, `APELADO`, `SUSPENDIDO`, `NO_SUSCRIPCION_CONTRATO POR_DECISION_ENTIDAD`, etc. |

- **`estado_adj`** (adjudicaciones): `CONTRATADO` 82%, `CONSENTIDO` 11%, `ADJUDICADO` 6%, `MIXTO` 0.5%.
- **`estado_contrato`** (contratos): `CONTRATADO` 82%, `sin_dato` 15%, `RESUELTO` 1.9% (2,897 contratos), `NULO` 0.3%.
- **`estado_proceso_grueso`** es la agrupación de la fuente: `complete`, `unsuccessful`, `active`, `cancelled` (o `MIXTO`).

### Diccionario: `procesos`
| Columna | Qué es |
|---|---|
| `release_id`, `ocid` | Identificadores del proceso (`ocid` es único) |
| `proceso_id` | Número del proceso en SEACE |
| `fecha_release` | Fecha de la última versión del proceso |
| `entidad_id`, `entidad` | Entidad contratante |
| `entidad_tipo` | Tipo de entidad deducido del nombre: `municipalidad`, `gobierno_regional`, `ministerio`, `universidad`, `salud`, `defensa_policia` u `otro`. **Heurístico** |
| `entidad_departamento`, `entidad_provincia`, `entidad_distrito` | Ubicación de la **sede** de la entidad (100% de cobertura). No es el lugar de ejecución |
| `titulo` | Código del proceso (por ejemplo `LP-ABR-6-2026-MDP/C-1`), no es descriptivo |
| `descripcion` | Descripción original del objeto |
| `texto` | `descripcion` normalizada (minúsculas, sin tildes ni símbolos). **Usar para similitud.** Hay muchas descripciones repetidas: 163,221 textos distintos entre 209,740 procesos. El 22% de los procesos son copias exactas de otro texto, y el 38% comparte su texto con al menos otro proceso |
| `n_palabras` | Número de palabras de `texto` |
| `categoria` | `goods`, `services` o `works` |
| `procedimiento` | Tipo de procedimiento (38 valores, más `sin_dato` en 1 proceso). **Usar este** |
| `metodo` | Categoría general (`open`, `direct`, `selective`) o `sin_dato` |
| `estado_proceso` | Cómo terminó o en qué está el proceso (ver "Estados"). `sin_dato` en 5 procesos |
| `estado_proceso_grueso` | Agrupación de la fuente: `complete`, `unsuccessful`, `active`, `cancelled` o `MIXTO` |
| `fecha_pub` | Fecha de publicación (hora de Lima). **Usar para ordenar el flujo** |
| `anio_pub`, `mes_pub`, `periodo` | Año, mes y `yyyy-MM` de `fecha_pub`. `anio_pub` es la columna de partición |
| `fecha_inicio_conv`, `fecha_fin_conv` | Inicio y fin de la convocatoria (6% nulos) |
| `valor_orig`, `moneda` | Valor referencial en su moneda original |
| `valor_pen` | Valor convertido a soles por la fuente. 0 = valor no disponible |
| `valor_cero` | `true` si `valor_pen` es 0 |
| `valor_pen_valido` | `valor_pen` sin ceros. **Usar para montos** |
| `info_protegida` | `true` si el proceso tiene información protegida por ley |
| `n_postores_declarado` | Postores según la fuente (10% nulos) |
| `n_postores` | Postores contados en la tabla `postores` |
| `proyecto_id` | Código del proyecto de inversión, si aplica (60% nulos) |
| `archivo_origen` | Año del zip de origen (año de última actualización) |
| `n_items`, `n_segmentos_unspsc` | Cantidad de ítems y de segmentos UNSPSC distintos |
| `n_adjudicaciones`, `n_proveedores_adj` | Adjudicaciones y proveedores distintos |
| `monto_adj_pen` | Suma de las adjudicaciones **en soles** (las de moneda extranjera se convierten con `tipo_cambio`). Nulo si no hay adjudicaciones con monto en soles |
| `n_adj_otra_moneda` | Adjudicaciones en otra moneda |
| `n_adj_sin_monto_pen` | Adjudicaciones sin monto en soles (sin tipo de cambio) |
| `n_contratos`, `n_contratos_sin_adj` | Contratos y los que no enlazan con una adjudicación |
| `n_contratos_resueltos` | Contratos con estado `RESUELTO` |
| `tiene_adjudicacion`, `tiene_contrato` | Indicadores |

### Diccionario: `items`
| Columna | Qué es |
|---|---|
| `release_id`, `ocid` | Identificadores del proceso (unen con `procesos`) |
| `item_id` | Identificador del ítem |
| `descripcion_item` | Descripción original del ítem |
| `texto_item` | `descripcion_item` normalizada (minúsculas, sin tildes ni símbolos) |
| `cantidad`, `unidad` | Cantidad y unidad de medida (por ejemplo "Servicio", "Unidad") |
| `valor_item`, `moneda_item` | Valor del ítem y su moneda original. Puede venir en 0 |
| `cubso_id`, `cubso_desc` | Clasificación peruana CUBSO. 16% nulos |
| `unspsc_id`, `unspsc_desc` | Clasificación internacional UNSPSC (8 dígitos). 16% nulos |
| `unspsc_segmento` | Los 2 primeros dígitos de `unspsc_id` (nivel más general) |
| `unspsc_familia` | Los 4 primeros dígitos de `unspsc_id` |
| `estado_item` | Estado general del ítem: `complete`, `unsuccessful`, `active` o `cancelled` |
| `estado_item_detalle` | Estado detallado del ítem (`CONTRATADO`, `DESIERTO`, `CONVOCADO`, ...) |

### Diccionario: `postores`
| Columna | Qué es |
|---|---|
| `release_id`, `ocid` | Identificadores del proceso (unen con `procesos`) |
| `ruc` | RUC del postor, sin el prefijo `PE-RUC-`. Los extranjeros empiezan con `L` |
| `postor` | Nombre o razón social |
| `extranjero` | `true` si el identificador empieza con `L` |

### Diccionario: `adjudicaciones`
| Columna | Qué es |
|---|---|
| `release_id`, `ocid` | Identificadores del proceso (unen con `procesos`) |
| `adj_id` | Identificador de la adjudicación. Une con `contratos.adj_id` |
| `fecha_adj` | Fecha de adjudicación (hora de Lima) |
| `monto`, `moneda` | Monto adjudicado en su moneda original |
| `tipo_cambio` | Soles por unidad de la moneda. Es 1 si ya está en soles. Nulo si la fuente no trae la tasa (95 casos) |
| `monto_pen` | `monto` × `tipo_cambio`. **Usar para sumar montos** |
| `proveedor_ruc`, `proveedor` | RUC y nombre del proveedor ganador. 1.8% sin proveedor |
| `proveedor_extranjero` | `true` si el RUC empieza con `L` |
| `estado_adj` | Estado de la adjudicación, deducido de sus ítems: `CONTRATADO`, `CONSENTIDO`, `ADJUDICADO` o `MIXTO` |
| `tiene_contrato` | `true` si algún contrato corresponde a esta adjudicación. Es `false` en el 16% |

### Diccionario: `contratos`
| Columna | Qué es |
|---|---|
| `release_id`, `ocid` | Identificadores del proceso (unen con `procesos`) |
| `contrato_id` | Identificador del contrato |
| `adj_id` | Adjudicación a la que corresponde |
| `fecha_firma` | Fecha de firma (hora de Lima) |
| `monto`, `moneda` | Monto del contrato en su moneda original, sin nulos |
| `tipo_cambio` | Soles por unidad de la moneda. Es 1 si ya está en soles. Si el contrato no trae tasa, se usa la de su adjudicación (misma moneda). Nulo en 76 casos |
| `monto_pen` | `monto` × `tipo_cambio`. **Usar para sumar montos** |
| `monto_final`, `moneda_final` | Monto final de ejecución y su moneda. Mayormente nulo (91%) |
| `inicio`, `fin`, `duracion_dias` | Periodo del contrato. `duracion_dias` casi siempre viene (0.8% nulos) |
| `estado_contrato` | `CONTRATADO`, `RESUELTO`, `NULO` o `sin_dato` (15%). Se deduce de los ítems del contrato |
| `adjudicacion_vinculada` | `false` si su `adj_id` no aparece en `adjudicaciones` (cerca del 1%) |

### Diccionario: `entidades`
| Columna | Qué es |
|---|---|
| `entidad_id`, `entidad` | Identificador de la entidad y su nombre más reciente |
| `entidad_tipo` | Tipo deducido del nombre (heurístico) |
| `entidad_departamento`, `entidad_provincia`, `entidad_distrito` | Ubicación de la sede |
| `n_departamentos` | Cuántos departamentos distintos tuvo la entidad en los datos (2 entidades tienen 2) |
| `n_procesos` | Procesos de la entidad en el periodo |